# Random Forest
Can a random forest beat the 64.8% logistic regression from 05? Same features and same test season, so the comparison is fair.

## Setup
Imports for loading the data, training the forest and scoring it.

In [1]:
import pandas as pd
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier

## Load the features
The same features file built in 04, with one row per team per game.

In [2]:
games = pd.read_csv('../data/features.csv')

## Pick the features
The same 12 features as 05: each team's last-10-game averages plus rest days. Trees don't care about scale, so unlike 05 there is no StandardScaler here.

In [3]:
features = ['ORTG_L10_HOME', 'DRTG_L10_HOME', 'REB_L10_HOME', 'WIN_L10_HOME', 'REST_DAYS_HOME', 'NET_L10_HOME',
            'ORTG_L10_AWAY', 'DRTG_L10_AWAY', 'REB_L10_AWAY', 'WIN_L10_AWAY', 'REST_DAYS_AWAY', 'NET_L10_AWAY']

## One row per game
Split the team rows into home and away, then merge them on GAME_ID so each row holds both teams side by side. Games where a team has fewer than 10 games of history have no averages yet, so drop them.

In [4]:
home = games[games['HOME'] == 1]
away = games[games['HOME'] == 0]

In [5]:
matchups = pd.merge(home, away, on='GAME_ID', suffixes=('_HOME', '_AWAY'))
matchups = matchups.dropna(subset=features)

## Inputs and target
X holds the 12 features and y is whether the home team won.

In [6]:
y = matchups['WIN_HOME']
X = matchups[features]

## Train on past seasons, test on 2025-26
Same time-based split as 05: the model never sees the 2025-26 season while training, so the test score shows how it would do on games it hasn't seen.

In [7]:
test_mask = matchups['SEASON_ID_HOME'] == 22025

In [8]:
X_train = X[~test_mask]
y_train = y[~test_mask]

X_test = X[test_mask]
y_test = y[test_mask]

In [9]:
X_train.shape, X_test.shape

((4291, 12), (1071, 12))

## Train the random forest
n_estimators is the number of trees (each built from its own bootstrapped sample of games). random_state makes the randomness repeatable, and oob_score tells it to also score each game using only the trees that never saw it.

In [10]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True)
rf_model.fit(X_train, y_train)

,"oob_score oob_score: bool or callable, default=FalseWhether to use out-of-bag samples to estimate the generalization score.By default, :func:`~sklearn.metrics.accuracy_score` is used.Provide a callable with signature `metric(y_true, y_pred)` to use acustom metric. Only available if `bootstrap=True`.For an illustration of out-of-bag (OOB) error estimation, see the example:ref:`sphx_glr_auto_examples_ensemble_plot_ensemble_oob.py`.",True
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the followi

## Predict the test season
Each test game goes down all 100 trees and the majority vote is the prediction.

In [11]:
preds = rf_model.predict(X_test)

## Test accuracy
63.2%. Better than always picking home (54.9%) but below logistic regression (64.8%). With default settings every tree grows until its leaves are pure, so each one overfits.

In [12]:
accuracy_score(y_test, preds)

0.6321195144724556

## Out-of-bag accuracy
60.5%: accuracy on the games each tree never saw. It runs a bit lower than the test score, but it lets us compare settings without touching the test season, which is how we'll tune the forest next.

In [13]:
rf_model.oob_score_

0.6049871824749475

## Tune min_samples_leaf
By default every tree grows until each leaf holds a single game, which overfits. Try requiring more games per leaf and keep whichever gives the best out-of-bag accuracy.

In [14]:
for leaf in [3, 5, 8, 10, 15, 20, 50, 100]:
    rf = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True, min_samples_leaf=leaf)
    rf.fit(X_train, y_train)
    print(leaf, rf.oob_score_)

3 0.6157072943369843


5 0.6098811465858774


8 0.6222325798182242


10 0.628757865299464


15 0.6240969470985784


20 0.6213003961780471


50 0.6206012584479143


100 0.6189699370776043


## Tune max_features
max_features is how many of the 12 features each split gets to choose from. The default is the square root of 12 (about 3), so try a few values on either side with min_samples_leaf fixed at 10.

In [15]:
for mf in [1, 2, 3, 4, 6]:
    rf = RandomForestClassifier(n_estimators=100, random_state=42, oob_score=True, min_samples_leaf=10, max_features=mf)
    rf.fit(X_train, y_train)
    print(mf, rf.oob_score_)

1 0.6245630389186669


2 0.6147751106968072


3 0.628757865299464


4 0.6196690748077371


6 0.6147751106968072


## Tuned random forest
The best settings were min_samples_leaf=10 and max_features=3. Use 300 trees so the vote is steadier, then score it on the 2025-26 season once.

In [16]:
rf_tuned = RandomForestClassifier(n_estimators=300, random_state=42, oob_score=True, min_samples_leaf=10, max_features=3)
rf_tuned.fit(X_train, y_train)
preds_tuned = rf_tuned.predict(X_test)
accuracy_score(y_test, preds_tuned)

0.6470588235294118

## What the forest relies on
Feature importance shows how much each feature helped the trees separate wins from losses.

In [17]:
pd.Series(rf_tuned.feature_importances_, index=features).sort_values(ascending=False)

NET_L10_HOME      0.160653
NET_L10_AWAY      0.132535
ORTG_L10_AWAY     0.113641
ORTG_L10_HOME     0.111774
DRTG_L10_HOME     0.099916
DRTG_L10_AWAY     0.095406
REB_L10_AWAY      0.075950
REB_L10_HOME      0.075775
WIN_L10_HOME      0.052318
WIN_L10_AWAY      0.048781
REST_DAYS_HOME    0.019849
REST_DAYS_AWAY    0.013404
dtype: float64

Net rating (points scored minus allowed per 100 possessions) for both teams matters most, followed by offensive and defensive ratings. Recent win % and rest days matter least, likely because net rating already captures how good a team has been.

## Results

| Model | Test accuracy (2025-26) |
|---|---|
| Always pick the home team | 54.9% |
| Random forest, default settings | 63.2% |
| Random forest, tuned | 64.7% |
| Logistic regression (05) | 64.8% |

Tuning helped the forest by 1.5 points, but it only ties logistic regression. The gap is one game out of 1,071, well inside the ±2.9% margin. A more complex model didn't find anything the simple one missed, so the next gains likely need better features (Elo, injuries, player data) rather than a fancier model.